# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

We use the March 2026 partition for iteration. June 2026 is the sealed final month.

The March parquet is downloaded through the authenticated Hugging Face account and then read locally by DuckDB.


In [16]:
from pathlib import Path
import duckdb
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
whoami = api.whoami()
print("Logged in as:", whoami["name"])

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
)

print("March file:", march_path)
print("Size (MB):", round(Path(march_path).stat().st_size / (1024 * 1024), 2))

con = duckdb.connect()


Logged in as: Parasiticwire
March file: C:\Users\zain\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet
Size (MB): 118.48


In [17]:
preview = con.sql(f'''
    SELECT *
    FROM read_parquet('{march_path}')
    LIMIT 5
''').df()

preview


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## Signal 1 check — Search volume

**Signal:** `gsc_impressions`

Higher observed search volume can make a content item more important to review because more search activity is associated with the item.

**Verdict:** based on the executed table below, record one of CONFIRMED, OPPOSITE, MIXED, or FALSE.


In [18]:
volume_check = con.sql(f'''
    WITH base AS (
        SELECT gsc_impressions, gsc_clicks
        FROM read_parquet('{march_path}')
        WHERE gsc_data_available IS TRUE
          AND gsc_impressions IS NOT NULL
    ),
    cutoffs AS (
        SELECT
            quantile_cont(gsc_impressions, 0.33) AS q33,
            quantile_cont(gsc_impressions, 0.66) AS q66
        FROM base
    )
    SELECT
        CASE
            WHEN gsc_impressions = 0 THEN '0'
            WHEN gsc_impressions <= q33 THEN 'Low'
            WHEN gsc_impressions <= q66 THEN 'Medium'
            ELSE 'High'
        END AS volume_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks
    FROM base
    CROSS JOIN cutoffs
    GROUP BY 1
    ORDER BY CASE volume_bucket
        WHEN '0' THEN 0 WHEN 'Low' THEN 1 WHEN 'Medium' THEN 2 ELSE 3 END
''').df()

volume_check


,volume_bucket,n,avg_impressions,avg_clicks
0,Low,1209356,2.76,0.01
1,Medium,1188669,17.65,0.04
2,High,1213036,211.32,0.63


### Signal 1 verdict: CONFIRMED

Your March result showed:

- Low: **1,209,356** rows, average **2.76** impressions and **0.01** clicks.
- Medium: **1,188,669** rows, average **17.65** impressions and **0.04** clicks.
- High: **1,213,036** rows, average **211.32** impressions and **0.63** clicks.

This supports using search volume as a prioritization signal. This is an observed association, not a causal claim.


## Signal 2 check — CTR relative to search position

CTR is calculated as `100 × clicks / impressions`.

Rows with unavailable GSC data, zero impressions, or `gsc_avg_position = 0` are excluded. In the FlyRank data, position `0` means no usable position data, not rank zero.


In [19]:
ctr_position_check = con.sql(f'''
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN 'Top (<=3)'
            WHEN gsc_avg_position <= 10 THEN 'Mid (3-10)'
            ELSE 'Lower (>10)'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(100.0 * gsc_clicks / NULLIF(gsc_impressions, 0)), 4) AS avg_ctr_pct,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0
    GROUP BY 1
    ORDER BY CASE position_bucket
        WHEN 'Top (<=3)' THEN 1
        WHEN 'Mid (3-10)' THEN 2
        ELSE 3
    END
''').df()

ctr_position_check


,position_bucket,n,avg_ctr_pct,avg_impressions,avg_clicks
0,Top (<=3),564173,0.4918,94.94,0.36
1,Mid (3-10),1456122,0.3473,94.66,0.31
2,Lower (>10),1427577,0.1828,62.20,0.12


### Signal 2 verdict

After running the table above, choose exactly one:

- **CONFIRMED** — observed pattern supports the signal.
- **OPPOSITE** — observed pattern clearly goes against it.
- **MIXED** — pattern varies by bucket.
- **FALSE** — no useful evidence.

Do not choose before looking at the output.


## 1. My rule and its reason codes

The rule uses only current-window, non-label-derived signals:

- impressions
- clicks
- observed CTR
- average position

It produces one numeric score, one reason code, and one action label.

**No future-window or label-derived inputs are used.**


In [20]:
baseline = con.sql(f'''
    WITH base AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            100.0 * gsc_clicks / NULLIF(gsc_impressions, 0) AS ctr_pct
        FROM read_parquet('{march_path}')
        WHERE gsc_data_available IS TRUE
          AND gsc_impressions > 0
          AND gsc_impressions IS NOT NULL
    ),
    scored AS (
        SELECT *,
            CASE
                WHEN gsc_avg_position > 3
                     AND gsc_avg_position <= 10
                     AND ctr_pct < 2 THEN 3.0
                WHEN gsc_avg_position > 10
                     AND gsc_impressions >= 50 THEN 2.0
                WHEN gsc_impressions >= 50 THEN 1.0
                ELSE 0.0
            END AS action_score
        FROM base
    )
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions AS impressions_30d,
        gsc_clicks AS clicks_30d,
        ROUND(ctr_pct, 4) AS ctr_pct,
        ROUND(gsc_avg_position, 2) AS position,
        action_score,
        CASE
            WHEN gsc_avg_position > 3 AND gsc_avg_position <= 10 AND ctr_pct < 2
                THEN 'LOW_CTR_MID_POSITION'
            WHEN gsc_avg_position > 10 AND gsc_impressions >= 50
                THEN 'HIGH_VOLUME_LOW_POSITION'
            WHEN gsc_impressions >= 50
                THEN 'MEANINGFUL_SEARCH_VOLUME'
            ELSE 'NO_STRONG_SIGNAL'
        END AS reason_code,
        CASE
            WHEN gsc_avg_position > 3 AND gsc_avg_position <= 10 AND ctr_pct < 2
                THEN 'Review CTR/title-snippet opportunity'
            WHEN gsc_avg_position > 10 AND gsc_impressions >= 50
                THEN 'Review content for ranking opportunity'
            WHEN gsc_impressions >= 50
                THEN 'Review high-demand content'
            ELSE 'Monitor'
        END AS action_label
    FROM scored
    ORDER BY action_score DESC, impressions_30d DESC
''').df()

baseline.head(20)


,report_date,client_hash_id,content_hash_id,impressions_30d,clicks_30d,ctr_pct,position,action_score,reason_code,action_label
0,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,0.0000,8.61,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
1,2026-03-22,client_62f4a7e64f5e0096,content_34a70fea29d15f24,27410,0,0.0000,3.13,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
2,2026-03-05,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,24456,1,0.0041,4.07,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
3,2026-03-16,client_62f4a7e64f5e0096,content_34a70fea29d15f24,19223,1,0.0052,3.63,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
4,2026-03-04,client_62f4a7e64f5e0096,content_60b99970e55b1ac5,16833,2,0.0119,3.91,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
5,2026-03-15,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,16454,1,0.0061,5.13,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
6,2026-03-28,client_23a62021009f63c4,content_e943d753806d7af3,15522,49,0.3157,8.79,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
7,2026-03-04,client_62f4a7e64f5e0096,content_465d20bc90052258,14367,12,0.0835,3.28,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
8,2026-03-16,client_e547b89c05043229,content_963de14b1f58978f,14274,168,1.1770,3.17,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity
9,2026-03-28,client_a80fca3f171ed1de,content_046fc480045b88f5,14185,1,0.0070,7.05,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity


## 2. Build the ranked queue (writes the CSV)

The required output is `work/outputs/baseline_action_score.csv`.

The internship instructions say this CSV stays out of git; the notebook regenerates it.

In [21]:
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
baseline.to_csv(output_path, index=False)

print("Wrote:", output_path)
print("Rows:", len(baseline))


Wrote: work\outputs\baseline_action_score.csv
Rows: 3611061


## 3. Top-20 review

Inspect the actual top 20. For every item, explain the action, why it was selected, and what could make the recommendation wrong.


In [22]:
top20 = baseline.head(20).copy()

top20[[
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "reason_code",
    "action_label",
    "impressions_30d",
    "clicks_30d",
    "ctr_pct",
    "position"
]]


,client_hash_id,content_hash_id,action_score,reason_code,action_label,impressions_30d,clicks_30d,ctr_pct,position
0,client_62f4a7e64f5e0096,content_945d6ff91386c817,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,37368,0,0.0000,8.61
1,client_62f4a7e64f5e0096,content_34a70fea29d15f24,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,27410,0,0.0000,3.13
2,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,24456,1,0.0041,4.07
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,19223,1,0.0052,3.63
4,client_62f4a7e64f5e0096,content_60b99970e55b1ac5,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,16833,2,0.0119,3.91
5,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,16454,1,0.0061,5.13
6,client_23a62021009f63c4,content_e943d753806d7af3,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,15522,49,0.3157,8.79
7,client_62f4a7e64f5e0096,content_465d20bc90052258,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,14367,12,0.0835,3.28
8,client_e547b89c05043229,content_963de14b1f58978f,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,14274,168,1.1770,3.17
9,client_a80fca3f171ed1de,content_046fc480045b88f5,3.0,LOW_CTR_MID_POSITION,Review CTR/title-snippet opportunity,14185,1,0.0070,7.05


## 4. Weak picks + leakage check


The baseline is intentionally simple. It can prioritize an item for reasons that are not visible in this data, including tracking problems, legitimate ranking states, client-specific context, or content that does not actually need a change.

The score is a prioritization signal, not proof that a content change will improve performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.